# Legacy regressor experiments (archived from 03_regressors.ipynb)

Multi-feature experiments (GPR / Ridge / Lasso / SVM) from the pre-refactor
`03_regressors.ipynb` (cells 12-45, season 24-25). Not part of the active pipeline;
kept as a reference for future model upgrades (roadmap Phase 4). These cells assume
the old 24-25 column names and the `squads.csv` file.

In [ ]:
# Create numeric copies of dataframes for exploratory ML models
dataframe_merge_num = dataframe_merge.copy()
dataframe_merge_num['squad'] = dataframe_merge_num['squad'].astype(str).str.upper().map(squads_dict).fillna(1)
dataframe_merge_num = dataframe_merge_num.fillna(0)

dataframe_A = dataframe_merge_num[(dataframe_merge_num['role'] == 'A') & (dataframe_merge_num['fvm'] >= 20)]
dataframe_C = dataframe_merge_num[dataframe_merge_num['role'] == 'C']
dataframe_D = dataframe_merge_num[dataframe_merge_num['role'] == 'D']
dataframe_P = dataframe_merge_num[dataframe_merge_num['role'] == 'P']


**GPR**

After inspecting the behaviour of this model I concluded it does not fit our problem.

In [12]:
# Get the data to fit
X = dataframe_A[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_A[["expprice", "expstd"]].values

# Define kernel
kernel = RationalQuadratic(length_scale=1)

# Fit the GP model
gp_model = GaussianProcessRegressor(kernel=kernel)

# define model evaluation method
cv = RepeatedKFold(n_splits=10, n_repeats=8, random_state=42)
# evaluate model
scores = cross_val_score(gp_model, X, y, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

gp_model.fit(X, y)

# Predict
row = [
        [5, 20, 37, 4.5, 5]
    ]
predicted_mean, predicted_std = gp_model.predict(row, return_std=True)

print(f"Predicted mean: {predicted_mean}")


Mean MSE: 423.472 (608.019)


c:\Users\d.cirino\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\_gpr.py:659: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


ValueError: X has 4 features, but GaussianProcessRegressor is expecting 3 features as input.

**Ridge**

Has the behaviour we want and from previous takes it performs better than Linear Regression

In [7]:
# Get the data to fit
X = dataframe_A[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_A[["expprice", "expstd"]].values

# Define model
model = Ridge(alpha=1.5)
# define model evaluation method
cv = RepeatedKFold(n_splits=10, n_repeats=8, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 20, 3.5, 5, 420]]
model.predict(row)

Mean MSE: 139.384 (108.956)


array([[334.06547347,  42.42942749]])

**Lasso**

Has the behaviour we want and from previous takes it performs better than Linear Regression

In [53]:
# Get the data to fit
X = dataframe_A[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_A[["expprice", "expstd"]].values

# Define model
model = Lasso(alpha=1)
# define model evaluation method
cv = RepeatedKFold(n_splits=10, n_repeats=8, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 40, 4.4, 5, 250]]
model.predict(row)

Mean MSE: 140.015 (111.970)


array([[135.08139252,  24.3643135 ]])

**Linear Regression**

In [440]:
# Get the data to fit
X = dataframe_A[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_A[["expprice", "expstd"]].values

# Create a linear regression model
model = LinearRegression()
# define model evaluation method
cv = RepeatedKFold(n_splits=10, n_repeats=8, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 20, 4.2, 4, 250]]
model.predict(row)

Mean MSE: 125.594 (97.359)


array([[167.70427997,  25.32717582]])

**SVM**

In [56]:
# Get the data to fit
X = dataframe_A[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_A[["expprice", "expstd"]].values

# Create an SVR model
svr = SVR(kernel='linear')  # You can choose different kernels like 'rbf', 'poly', etc.

# Wrap SVR into MultiOutputRegressor
model = MultiOutputRegressor(svr)
# define model evaluation method
cv = RepeatedKFold(n_splits=10, n_repeats=8, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[4, 50, 3.5, 5, 200]]
model.predict(row)

Mean MSE: 147.597 (130.852)


array([[92.28380825, 19.98913454]])

## GPR Regressors

**Attackers**

In [575]:
data = dataframe_A.values
# Get the data to fit
X = dataframe_A[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_A[["expprice", "expstd"]].values

# Scale the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Define kernel
kernel = RationalQuadratic(length_scale=0.1)

# Fit the GP model
model = GaussianProcessRegressor(kernel=kernel)
# define model evaluation method
cv = RepeatedKFold(n_splits=10, n_repeats=8, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[4, 30, 4.6, 5, 200]]
row = scaler.transform(row)
model.predict(row)

temp = model.predict(X)
temp = np.round(temp)
temp = temp.astype(int)
#temp = np.concatenate((['price'] , temp), axis=0)
temp = np.vstack(temp)
tempJoint = np.hstack((data, temp))
print(tempJoint[20:80,:])
name = 'prices_A.csv'
print(name)
np.savetxt(name, tempJoint, delimiter=',', fmt='%s')
nameEx = 'prices_A_ex.csv'
np.savetxt(nameEx, tempJoint, delimiter=';', fmt='%s')
# Save the model to a file
joblib.dump(model, 'models/multi_feature/lasso_A.pkl')

Mean MSE: 313.279 (334.262)
[['A' 'Beltran L.' 4 15 3.0 3 61 14.07142857142857 7.670766541921304 14
  14 8]
 ['A' 'Brekalo' 4 11 2.7 3 17 1.0 1.0 1 1 1]
 ['A' "Kouame'" 4 8 2.0 3 8 1.0 0.0 2 1 0]
 ['A' 'Cheddira' 2 10 2.0 5 28 5.538461538461538 4.858247019508591 13 6 5]
 ['A' 'Caso' 2 6 2.0 3 12 1.0 1.0 0 1 1]
 ['A' "Soule'" 2 4 2.0 4 14 1.571428571428571 0.9759000729485332 7 2 1]
 ['A' 'Cuni' 2 4 1.0 2 5 1.0 1.0 0 1 1]
 ['A' 'Kvernadze' 2 3 1.0 1 6 1.0 1.0 0 1 1]
 ['A' 'Kaio Jorge' 2 1 1.0 1 9 1.0 0.0 6 1 0]
 ['A' 'Bidaoui' 2 1 1.0 1 1 1.0 1.0 0 1 1]
 ['A' 'Retegui' 3 23 4.0 5 128 64.71428571428571 15.27908863285207 14 65
  15]
 ['A' 'Ekuban' 3 3 1.0 2 2 1.0 1.0 0 1 1]
 ['A' 'Puscas' 3 1 1.0 1 2 1.0 1.0 0 1 1]
 ['A' 'Yalcin' 3 1 1.0 1 1 1.0 1.0 0 1 1]
 ['A' 'Martinez L.' 5 40 4.8 5 436 331.6428571428572 38.15677607502447 14
  332 38]
 ['A' 'Thuram' 5 25 4.0 4 180 131.9285714285714 20.5817853965561 14 132
  21]
 ['A' 'Arnautovic' 5 23 3.6 3 122 32.78571428571428 15.37319631160563 14
  

NameError: name 'can' is not defined

## Ridge Regressors

**Attackers**

In [191]:
# Get the data to fit
X = dataframe_A[["squad", "myrating", "fvm"]].values
y = dataframe_A[["expprice", "expstd"]].values

# Define model
model = Ridge(alpha=1)
# define model evaluation method
cv = RepeatedKFold(n_splits=4, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

# Save the model to a file
joblib.dump(model, 'models/multi_feature/ridge_A.pkl')

Mean MSE: 1.612 (0.249)


['ridge_regressor_model_A.pkl']

In [193]:
row = [[5, 4.6, 100]]
print(model.predict(row))

[[50.70255599 13.5998253 ]]


**Midfielders**

In [99]:
# Get the data to fit
X = dataframe_C[["squad", "myrating", "fvm"]].values
y = dataframe_C[["expprice"]].values

# Define model
model = Ridge(alpha=1.5)
# define model evaluation method
cv = RepeatedKFold(n_splits=8, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[4, 4.5, 7.5]]
print(model.predict(row))

# Save the model to a file
joblib.dump(model, 'models/multi_feature/ridge_C.pkl')

Mean MSE: 4.813 (0.741)
[[ 7.38371268 49.51152719]]


['ridge_regressor_model_C.pkl']

**Defenders**

In [84]:
# Get the data to fit
X = dataframe_D[["squad", "myrating", "fvm"]].values
y = dataframe_D[["expprice"]].values

# Define model
model = Ridge(alpha=1.5)
# define model evaluation method
cv = RepeatedKFold(n_splits=4, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 4, 6.5]]
model.predict(row)

# Save the model to a file
joblib.dump(model, 'models/multi_feature/ridge_D.pkl')

Mean MSE: 3.028 (0.490)


['ridge_regressor_model_D.pkl']

**Keepers**

In [86]:
# Get the data to fit
X = dataframe_P[["squad", "myrating", "fvm"]].values
y = dataframe_P[["expprice"]].values

# Define model
model = Ridge(alpha=1.5)
# define model evaluation method
cv = RepeatedKFold(n_splits=4, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 4, 5.2]]
model.predict(row)
# Save the model to a file
joblib.dump(model, 'models/multi_feature/ridge_P.pkl')

Mean MSE: 3.787 (0.784)


['ridge_regressor_model_P.pkl']

## Lasso Regressors

**Attackers**

In [89]:
# Get the data to fit
X = dataframe_A[["squad", "myrating", "fvm"]].values
y = dataframe_A[["expprice"]].values

# Define model
model = Lasso(alpha=1)
# define model evaluation method
cv = RepeatedKFold(n_splits=4, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[4, 4.1, 7.5]]
model.predict(row)

# Save the model to a file
joblib.dump(model, 'models/multi_feature/lasso_A.pkl')

Mean MSE: 14.626 (2.597)


['lasso_regressor_model_A.pkl']

In [91]:
row = [[4, 4.1, 7.5]]
model.predict(row)

array([[ 6.88475248, 85.51941681]])

**Midfielders**

In [96]:
# Get the data to fit
X = dataframe_C[["squad", "myrating", "fvm"]].values
y = dataframe_C[["expprice"]].values

# Define model
model = Lasso(alpha=1)
# define model evaluation method
cv = RepeatedKFold(n_splits=8, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 5, 8.4]]
print(model.predict(row))

# Save the model to a file
joblib.dump(model, 'models/multi_feature/lasso_C.pkl')

Mean MSE: 4.599 (0.861)
[[ 6.35587209 54.90152561]]


['lasso_regressor_model_C.pkl']

**Defenders**

In [60]:
# Get the data to fit
X = dataframe_D[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_D[["expprice"]].values

# Define model
model = Lasso(alpha=1)
# define model evaluation method
cv = RepeatedKFold(n_splits=4, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 15, 4, 5, 50]]
model.predict(row)

# Save the model to a file
joblib.dump(model, 'models/multi_feature/lasso_D.pkl')

Mean MSE: 2.194 (0.286)


['lasso_regressor_model_D.pkl']

**Keepers**

In [61]:
# Get the data to fit
X = dataframe_P[["squad", "price", "myrating", "regularness", "fvm"]].values
y = dataframe_P[["expprice"]].values

# Define model
model = Lasso(alpha=1)
# define model evaluation method
cv = RepeatedKFold(n_splits=4, n_repeats=3, random_state=42)
# evaluate model
scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
# force scores to be positive
scores = np.abs(scores)
print('Mean MSE: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))
model.fit(X, y)

row = [[5, 20, 4, 4, 80]]
model.predict(row)
# Save the model to a file
joblib.dump(model, 'models/multi_feature/lasso_P.pkl')

Mean MSE: 1.884 (0.376)


['lasso_regressor_model_P.pkl']